In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [2]:
import pandas as pd
from statsmodels.formula import api as smf
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm


from data_helpers.preprocessors.scalers import denormalize_feats, normalize_feats, normalize_other

In [3]:
model_name = 'ob_rlm'

## Load and Filter Dataset

In [4]:
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft').reset_index(drop=True)
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Keep relevant columns and prepare train-test loader

In [5]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
feature_cols = [col for col in time_aggregated_dataset.columns.values if "change" not in col and "running_" in col] + ['realized_price']# + ['n_deal_prices_round'] + ['round']
rounds = range(1,5)
n_deal_prices = range(0,6)
min_quantile = 0.35
max_quantile = 1 - min_quantile
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)

## Fit and evaluate models

In [6]:
np.random.seed(1)
all_results = []
regression_res = []

for i in tqdm(range(pdl.max_samples)):
    train_df, test_df = pdl.get_sample_split_dataset(i)
    for rd in rounds:
        train_query = 'round <= ' + str(rd) # model performance DROPS if we include all the dataset
        formula = 'allocative_efficiency_round~ feedback_setting:('+'+'.join(feature_cols)+'+ n_unique_deals_round)' # including feedback setting as feedbackse∈g:(featcols...feedback_setting:(feat_cols...) degrades performance by 1%.

        train_df = train_df.copy()
        # normalize price features
        X_norm, _, _ = normalize_feats(train_df[feature_cols].values, min_quantile=min_quantile, max_quantile=max_quantile)
        train_df[feature_cols] = X_norm

        model = smf.rlm(formula, data=train_df.query(train_query))
        best_model = model.fit()

        # save the model parameters
        a = (best_model.summary2().tables[1]).stack().to_frame().T.swaplevel(-2, -1, axis=1)
        a['sample_id'] = i
        a['round'] = rd
        regression_res.append(a)
        
        for n_deal_price in n_deal_prices:
            key = (rd, n_deal_price)
            # pick subtest set
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query).copy()

            # normalize price feats
            X_test_norm, _, _ = normalize_feats(sub_test_set[feature_cols].values, min_quantile=min_quantile, max_quantile=max_quantile)
            sub_test_set[feature_cols] = X_test_norm

            # predict on the test set
            prediction = best_model.predict(sub_test_set)  
            prediction = np.clip(prediction, a_min=0, a_max=1.0)
            test_targets = sub_test_set['allocative_efficiency_round']

            # Calculate adjusted MAPE w.r.t. to zero targets
            result_test_df = sub_test_set[key_columns].copy()
            denom = test_targets.copy()
            denom[denom==0]= prediction[denom==0]
            denom[denom==0] = 1

            # save test errors
            result_test_df.loc[:, 'ae_ape'] = (np.abs(prediction - test_targets)/denom)
            result_test_df.loc[:, 'sample_id'] = i
            all_results.append(result_test_df)

  0%|          | 0/50 [00:00<?, ?it/s]

## Combine results to dataframes

In [7]:
all_results_df = pd.concat(all_results, ignore_index = True)
all_results_df['model'] = model_name
regression_data_df = pd.concat(regression_res, axis=0, ignore_index = True)

## Save results to files

In [8]:
regression_data_df.to_feather('../../../data/results/allocative_efficiency/'+model_name+'_data.ft')
all_results_df.to_feather('../../../data/results/allocative_efficiency/'+model_name+'.ft')